In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub

# Download latest version
path = kagglehub.competition_download('playground-series-s6e9')

print("Path to competition files:", path)
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e9/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e9/train.csv
/kaggle/input/competitions/playground-series-s6e9/test.csv
Path to competition files: /kaggle/input/competitions/playground-series-s6e9


In [2]:
train= pd.read_csv("/kaggle/input/competitions/playground-series-s6e9/train.csv")
test= pd.read_csv("/kaggle/input/competitions/playground-series-s6e9/test.csv")
train.head()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes
3,3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
4,4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No


In [3]:
test.head()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level
0,668665,61,67725.0,16.9,2,7,4,4.0,Male,Suburban,Sedan,Yes,No,Low
1,668666,42,152835.0,41.9,2,9,9,4.0,Male,Urban,SUV,No,No,Low
2,668667,68,86877.0,53.3,1,10,11,4.0,Female,Urban,Sedan,No,No,Low
3,668668,39,46794.0,34.1,2,4,8,4.0,Female,Suburban,Sedan,Yes,No,Low
4,668669,55,112172.0,57.2,1,6,4,1.0,Female,Suburban,Sedan,Yes,Yes,Low


In [4]:
train= train.drop(columns="id")
test= test.drop(columns="id")
train.head()

,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes
3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No


## Missing Values

In [5]:
train.isnull().mean()*100

Age                            0.0
Annual_Income_USD              0.0
Daily_Commute_km               0.0
Number_of_Cars_Owned           0.0
Charging_Stations_Near_Home    0.0
Charging_Stations_Near_Work    0.0
Environmental_Concern_Level    0.0
Gender                         0.0
City_Type                      0.0
Current_Car_Type               0.0
Home_Charging_Possible         0.0
Subsidy_Available              0.0
Range_Anxiety_Level            0.0
Will_Buy_EV                    0.0
dtype: float64

In [6]:
test.isnull().mean()*100

Age                            0.0
Annual_Income_USD              0.0
Daily_Commute_km               0.0
Number_of_Cars_Owned           0.0
Charging_Stations_Near_Home    0.0
Charging_Stations_Near_Work    0.0
Environmental_Concern_Level    0.0
Gender                         0.0
City_Type                      0.0
Current_Car_Type               0.0
Home_Charging_Possible         0.0
Subsidy_Available              0.0
Range_Anxiety_Level            0.0
dtype: float64

## Encoding

In [7]:
train["Gender"].unique()

array(['Male', 'Female', 'Other'], dtype=object)

In [8]:
print(train["Number_of_Cars_Owned"].unique())
print(train["Environmental_Concern_Level"].unique())
print(train["Charging_Stations_Near_Home"].unique())
print(train["Charging_Stations_Near_Work"].unique())
print(train["City_Type"].unique())
print(train["Current_Car_Type"].unique())

[2 1 4 3]
[1. 4. 5. 3. 2.]
[ 3  2  8  6  1 13 11  0  9  7 10  4  5 14 12]
[ 7  2 15  9  3  5  1 17 18 11 16 12 14 13 10  0  8 19  4  6]
['Suburban' 'Rural' 'Urban']
['Sedan' 'SUV' 'Hatchback' 'Truck']


In [9]:
## Using OneHotEncoder for Gender, City_Type and Current_Car_Type
train = pd.get_dummies(train, columns=["Gender","City_Type","Current_Car_Type"], prefix=["Gender","City_Type","Current_Car_Type"], dtype=int)

In [10]:
train

,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,...,Gender_Female,Gender_Male,Gender_Other,City_Type_Rural,City_Type_Suburban,City_Type_Urban,Current_Car_Type_Hatchback,Current_Car_Type_SUV,Current_Car_Type_Sedan,Current_Car_Type_Truck
0,66,92887.0,23.4,2,3,7,1.0,Yes,No,Low,...,0,1,0,0,1,0,0,0,1,0
1,38,30000.0,5.0,1,2,2,4.0,Yes,No,Low,...,0,1,0,1,0,0,0,1,0,0
2,26,94389.0,36.8,1,8,15,5.0,No,Yes,Low,...,1,0,0,0,0,1,0,0,1,0
3,66,73580.0,23.7,2,6,9,3.0,Yes,No,Low,...,0,1,0,0,1,0,1,0,0,0
4,54,57898.0,50.8,1,2,3,3.0,Yes,No,Low,...,0,1,0,0,1,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
668660,30,71090.0,27.4,2,5,6,5.0,No,Yes,Medium,...,0,1,0,0,0,1,0,0,1,0
668661,32,129990.0,5.0,2,5,4,1.0,Yes,Yes,Low,...,1,0,0,0,1,0,0,1,0,0
668662,64,121791.0,24.2,1,5,6,1.0,Yes,Yes,Low,...,1,0,0,0,1,0,1,0,0,0
668663,51,115923.0,43.7,2,0,0,1.0,Yes,Yes,Low,...,1,0,0,1,0,0,0,1,0,0


In [11]:
## Using Ordinal Encoder for Home_Charging_Possible,Subsidy_Available	and Range_Anxiety_Level
cat=["No","Yes"]
from sklearn.preprocessing import OrdinalEncoder
enc= OrdinalEncoder(categories=[cat])
train["Home_Charging_Possible"]= enc.fit_transform(train[["Home_Charging_Possible"]])
train["Subsidy_Available"]= enc.fit_transform(train[["Subsidy_Available"]])
train['Will_Buy_EV']= enc.fit_transform(train[['Will_Buy_EV']])
train

,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,...,Gender_Female,Gender_Male,Gender_Other,City_Type_Rural,City_Type_Suburban,City_Type_Urban,Current_Car_Type_Hatchback,Current_Car_Type_SUV,Current_Car_Type_Sedan,Current_Car_Type_Truck
0,66,92887.0,23.4,2,3,7,1.0,1.0,0.0,Low,...,0,1,0,0,1,0,0,0,1,0
1,38,30000.0,5.0,1,2,2,4.0,1.0,0.0,Low,...,0,1,0,1,0,0,0,1,0,0
2,26,94389.0,36.8,1,8,15,5.0,0.0,1.0,Low,...,1,0,0,0,0,1,0,0,1,0
3,66,73580.0,23.7,2,6,9,3.0,1.0,0.0,Low,...,0,1,0,0,1,0,1,0,0,0
4,54,57898.0,50.8,1,2,3,3.0,1.0,0.0,Low,...,0,1,0,0,1,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
668660,30,71090.0,27.4,2,5,6,5.0,0.0,1.0,Medium,...,0,1,0,0,0,1,0,0,1,0
668661,32,129990.0,5.0,2,5,4,1.0,1.0,1.0,Low,...,1,0,0,0,1,0,0,1,0,0
668662,64,121791.0,24.2,1,5,6,1.0,1.0,1.0,Low,...,1,0,0,0,1,0,1,0,0,0
668663,51,115923.0,43.7,2,0,0,1.0,1.0,1.0,Low,...,1,0,0,1,0,0,0,1,0,0


In [12]:
cat=["Low","Medium","High"]
enc= OrdinalEncoder(categories=[cat])
train["Range_Anxiety_Level"]= enc.fit_transform(train[["Range_Anxiety_Level"]])
train

,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,...,Gender_Female,Gender_Male,Gender_Other,City_Type_Rural,City_Type_Suburban,City_Type_Urban,Current_Car_Type_Hatchback,Current_Car_Type_SUV,Current_Car_Type_Sedan,Current_Car_Type_Truck
0,66,92887.0,23.4,2,3,7,1.0,1.0,0.0,0.0,...,0,1,0,0,1,0,0,0,1,0
1,38,30000.0,5.0,1,2,2,4.0,1.0,0.0,0.0,...,0,1,0,1,0,0,0,1,0,0
2,26,94389.0,36.8,1,8,15,5.0,0.0,1.0,0.0,...,1,0,0,0,0,1,0,0,1,0
3,66,73580.0,23.7,2,6,9,3.0,1.0,0.0,0.0,...,0,1,0,0,1,0,1,0,0,0
4,54,57898.0,50.8,1,2,3,3.0,1.0,0.0,0.0,...,0,1,0,0,1,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
668660,30,71090.0,27.4,2,5,6,5.0,0.0,1.0,1.0,...,0,1,0,0,0,1,0,0,1,0
668661,32,129990.0,5.0,2,5,4,1.0,1.0,1.0,0.0,...,1,0,0,0,1,0,0,1,0,0
668662,64,121791.0,24.2,1,5,6,1.0,1.0,1.0,0.0,...,1,0,0,0,1,0,1,0,0,0
668663,51,115923.0,43.7,2,0,0,1.0,1.0,1.0,0.0,...,1,0,0,1,0,0,0,1,0,0


In [13]:
## Encoding for test data
## Using OneHotEncoder for Gender, City_Type and Current_Car_Type
test = pd.get_dummies(test, columns=["Gender","City_Type","Current_Car_Type"], prefix=["Gender","City_Type","Current_Car_Type"], dtype=int)
## Using Ordinal Encoder for Home_Charging_Possible,Subsidy_Available	and Range_Anxiety_Level
cat=["No","Yes"]
from sklearn.preprocessing import OrdinalEncoder
enc= OrdinalEncoder(categories=[cat])
test["Home_Charging_Possible"]= enc.fit_transform(test[["Home_Charging_Possible"]])
test["Subsidy_Available"]= enc.fit_transform(test[["Subsidy_Available"]])
cat=["Low","Medium","High"]
enc= OrdinalEncoder(categories=[cat])
test["Range_Anxiety_Level"]= enc.fit_transform(test[["Range_Anxiety_Level"]])
test

,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Gender_Female,Gender_Male,Gender_Other,City_Type_Rural,City_Type_Suburban,City_Type_Urban,Current_Car_Type_Hatchback,Current_Car_Type_SUV,Current_Car_Type_Sedan,Current_Car_Type_Truck
0,61,67725.0,16.9,2,7,4,4.0,1.0,0.0,0.0,0,1,0,0,1,0,0,0,1,0
1,42,152835.0,41.9,2,9,9,4.0,0.0,0.0,0.0,0,1,0,0,0,1,0,1,0,0
2,68,86877.0,53.3,1,10,11,4.0,0.0,0.0,0.0,1,0,0,0,0,1,0,0,1,0
3,39,46794.0,34.1,2,4,8,4.0,1.0,0.0,0.0,1,0,0,0,1,0,0,0,1,0
4,55,112172.0,57.2,1,6,4,1.0,1.0,1.0,0.0,1,0,0,0,1,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286566,63,75830.0,23.9,1,3,3,1.0,1.0,0.0,0.0,0,1,0,0,1,0,0,1,0,0
286567,27,79563.0,55.2,2,7,5,3.0,1.0,1.0,0.0,0,1,0,0,1,0,0,0,1,0
286568,30,48183.0,42.6,3,7,7,4.0,1.0,0.0,0.0,1,0,0,0,1,0,0,0,1,0
286569,32,107845.0,5.0,2,14,7,1.0,0.0,0.0,0.0,1,0,0,0,0,1,0,1,0,0


## Scaling

In [14]:
train.columns

Index(['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned',
       'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work',
       'Environmental_Concern_Level', 'Home_Charging_Possible',
       'Subsidy_Available', 'Range_Anxiety_Level', 'Will_Buy_EV',
       'Gender_Female', 'Gender_Male', 'Gender_Other', 'City_Type_Rural',
       'City_Type_Suburban', 'City_Type_Urban', 'Current_Car_Type_Hatchback',
       'Current_Car_Type_SUV', 'Current_Car_Type_Sedan',
       'Current_Car_Type_Truck'],
      dtype='object')

In [15]:
X= train.drop(columns="Will_Buy_EV")
y= train["Will_Buy_EV"]
X

,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Gender_Female,Gender_Male,Gender_Other,City_Type_Rural,City_Type_Suburban,City_Type_Urban,Current_Car_Type_Hatchback,Current_Car_Type_SUV,Current_Car_Type_Sedan,Current_Car_Type_Truck
0,66,92887.0,23.4,2,3,7,1.0,1.0,0.0,0.0,0,1,0,0,1,0,0,0,1,0
1,38,30000.0,5.0,1,2,2,4.0,1.0,0.0,0.0,0,1,0,1,0,0,0,1,0,0
2,26,94389.0,36.8,1,8,15,5.0,0.0,1.0,0.0,1,0,0,0,0,1,0,0,1,0
3,66,73580.0,23.7,2,6,9,3.0,1.0,0.0,0.0,0,1,0,0,1,0,1,0,0,0
4,54,57898.0,50.8,1,2,3,3.0,1.0,0.0,0.0,0,1,0,0,1,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
668660,30,71090.0,27.4,2,5,6,5.0,0.0,1.0,1.0,0,1,0,0,0,1,0,0,1,0
668661,32,129990.0,5.0,2,5,4,1.0,1.0,1.0,0.0,1,0,0,0,1,0,0,1,0,0
668662,64,121791.0,24.2,1,5,6,1.0,1.0,1.0,0.0,1,0,0,0,1,0,1,0,0,0
668663,51,115923.0,43.7,2,0,0,1.0,1.0,1.0,0.0,1,0,0,1,0,0,0,1,0,0


In [16]:
## Using Standard Scaler to scale down all features in train
from sklearn.preprocessing import StandardScaler
std_scaler= StandardScaler()
feature_names = X.columns 
X_scaled = std_scaler.fit_transform(X)
X = pd.DataFrame(X_scaled, columns=feature_names)
X

,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Gender_Female,Gender_Male,Gender_Other,City_Type_Rural,City_Type_Suburban,City_Type_Urban,Current_Car_Type_Hatchback,Current_Car_Type_SUV,Current_Car_Type_Sedan,Current_Car_Type_Truck
0,1.472636,0.283361,-0.467597,0.394055,-0.499233,-0.033994,-1.354316,0.667240,-1.299244,-0.321865,-0.889677,0.904020,-0.089248,-0.477100,1.272142,-0.873277,-0.367175,-0.764241,1.097031,-0.249628
1,-0.702048,-1.911800,-1.449954,-0.977170,-0.753891,-0.998012,0.744881,0.667240,-1.299244,-0.321865,-0.889677,0.904020,-0.089248,2.095996,-0.786076,-0.873277,-0.367175,1.308488,-0.911551,-0.249628
2,-1.634055,0.335791,0.247816,-0.977170,0.774056,1.508436,1.444613,-1.498711,0.769678,-0.321865,1.124004,-1.106171,-0.089248,-0.477100,-0.786076,1.145112,-0.367175,-0.764241,1.097031,-0.249628
3,1.472636,-0.390577,-0.451580,0.394055,0.264740,0.351613,0.045149,0.667240,-1.299244,-0.321865,-0.889677,0.904020,-0.089248,-0.477100,1.272142,-0.873277,2.723499,-0.764241,-0.911551,-0.249628
4,0.540629,-0.937980,0.995261,-0.977170,-0.753891,-0.805209,0.045149,0.667240,-1.299244,-0.321865,-0.889677,0.904020,-0.089248,-0.477100,1.272142,-0.873277,2.723499,-0.764241,-0.911551,-0.249628
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
668660,-1.323386,-0.477495,-0.254041,0.394055,0.010082,-0.226798,1.444613,-1.498711,0.769678,2.895801,-0.889677,0.904020,-0.089248,-0.477100,-0.786076,1.145112,-0.367175,-0.764241,1.097031,-0.249628
668661,-1.168051,1.578495,-1.449954,0.394055,0.010082,-0.612405,-1.354316,0.667240,0.769678,-0.321865,1.124004,-1.106171,-0.089248,-0.477100,1.272142,-0.873277,-0.367175,1.308488,-0.911551,-0.249628
668662,1.317301,1.292297,-0.424885,-0.977170,0.010082,-0.226798,-1.354316,0.667240,0.769678,-0.321865,1.124004,-1.106171,-0.089248,-0.477100,1.272142,-0.873277,2.723499,-0.764241,-0.911551,-0.249628
668663,0.307627,1.087466,0.616200,0.394055,-1.263206,-1.383620,-1.354316,0.667240,0.769678,-0.321865,1.124004,-1.106171,-0.089248,2.095996,-0.786076,-0.873277,-0.367175,1.308488,-0.911551,-0.249628


In [17]:
## Using Standard Scaler to scale down all features in test
from sklearn.preprocessing import StandardScaler
std_scaler= StandardScaler()
feature_names = test.columns 
test_scaled = std_scaler.fit_transform(test)
test = pd.DataFrame(test_scaled, columns=feature_names)
test

,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Gender_Female,Gender_Male,Gender_Other,City_Type_Rural,City_Type_Suburban,City_Type_Urban,Current_Car_Type_Hatchback,Current_Car_Type_SUV,Current_Car_Type_Sedan,Current_Car_Type_Truck
0,1.082733,-0.596452,-0.814942,0.388333,0.521635,-0.608387,0.746352,0.666204,-1.303298,-0.318774,-0.890037,0.904324,-0.089058,-0.477967,1.269995,-0.870884,-0.367536,-0.764382,1.096020,-0.247928
1,-0.394162,2.376636,0.520214,0.388333,1.030720,0.355585,0.746352,-1.501041,-1.303298,-0.318774,-0.890037,0.904324,-0.089058,-0.477967,-0.787405,1.148259,-0.367536,1.308246,-0.912392,-0.247928
2,1.626852,0.072572,1.129045,-0.979493,1.285262,0.741174,0.746352,-1.501041,-1.303298,-0.318774,1.123548,-1.105798,-0.089058,-0.477967,-0.787405,1.148259,-0.367536,-0.764382,1.096020,-0.247928
3,-0.627356,-1.327620,0.103645,0.388333,-0.241992,0.162791,0.746352,0.666204,-1.303298,-0.318774,1.123548,-1.105798,-0.089058,-0.477967,1.269995,-0.870884,-0.367536,-0.764382,1.096020,-0.247928
4,0.616345,0.956184,1.337329,-0.979493,0.267093,-0.608387,-1.355167,0.666204,0.767284,-0.318774,1.123548,-1.105798,-0.089058,-0.477967,1.269995,-0.870884,-0.367536,-0.764382,1.096020,-0.247928
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286566,1.238196,-0.313326,-0.441098,-0.979493,-0.496534,-0.801182,-1.355167,0.666204,-1.303298,-0.318774,-0.890037,0.904324,-0.089058,-0.477967,1.269995,-0.870884,-0.367536,1.308246,-0.912392,-0.247928
286567,-1.560132,-0.182923,1.230517,0.388333,0.521635,-0.415593,0.045846,0.666204,0.767284,-0.318774,-0.890037,0.904324,-0.089058,-0.477967,1.269995,-0.870884,-0.367536,-0.764382,1.096020,-0.247928
286568,-1.326938,-1.279099,0.557598,1.756160,0.521635,-0.030004,0.746352,0.666204,-1.303298,-0.318774,1.123548,-1.105798,-0.089058,-0.477967,1.269995,-0.870884,-0.367536,-0.764382,1.096020,-0.247928
286569,-1.171475,0.805032,-1.450476,0.388333,2.303432,-0.030004,-1.355167,-1.501041,-1.303298,-0.318774,1.123548,-1.105798,-0.089058,-0.477967,-0.787405,1.148259,-0.367536,1.308246,-0.912392,-0.247928


## Choosing best model

In [18]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test= train_test_split(X,y,random_state=42, test_size=0.2)

In [19]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
import xgboost as xgb
import lightgbm as lgb

models={
    "lr": LogisticRegression(),
    "dtree": DecisionTreeClassifier(max_depth=15),
    "rf": RandomForestClassifier(n_estimators=100, max_depth=15, n_jobs=-1),
    "ada": AdaBoostClassifier(n_estimators=50), 
    "xg": xgb.XGBClassifier(n_jobs=-1),
    "lgb": lgb.LGBMClassifier(n_jobs=-1),
}
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring="roc_auc")
    print(f"{name} AUC: {scores.mean()*100:.4f}%")

lr AUC: 93.8033%
dtree AUC: 91.5575%
rf AUC: 93.8504%
ada AUC: 93.5844%
xg AUC: 94.1053%
[LightGBM] [Info] Number of positive: 74868, number of negative: 353077
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.024105 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 644
[LightGBM] [Info] Number of data points in the train set: 427945, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.174948 -> initscore=-1.550960
[LightGBM] [Info] Start training from score -1.550960
[LightGBM] [Info] Number of positive: 74869, number of negative: 353076
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.021548 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 644
[LightGBM] [Info] Num

## Using XGBoost for model training

In [21]:
import xgboost as xgb
model= xgb.XGBClassifier(n_jobs=-1)
model.fit(X_train,y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=-1, num_parallel_tree=None, ...)

In [22]:
## Testing model on X_test and y_test
from sklearn.metrics import roc_auc_score
y_pred_proba = model.predict_proba(X_test)[:, 1]
auc_score = roc_auc_score(y_test, y_pred_proba)
print(f"AUC Score: {auc_score:.5f}")

AUC Score: 0.94147


In [23]:
## Training model again on whole train dataset
model.fit(X,y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=-1, num_parallel_tree=None, ...)

## Creating submissions.csv

In [27]:
## Predicting probabilities for test.csv
org_test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e9/sample_submission.csv")
pred = model.predict_proba(test)[:, 1]
submission = pd.DataFrame({
    "id": org_test["id"],
    "Will_Buy_EV": pred
})
submission.to_csv("submission.csv", index=False)
submission

,id,Will_Buy_EV
0,668665,0.008853
1,668666,0.015618
2,668667,0.004386
3,668668,0.001810
4,668669,0.000207
...,...,...
286566,955231,0.000082
286567,955232,0.003044
286568,955233,0.002842
286569,955234,0.000481
